# EDA of North Atlantic Basin part of ME4OH dataset - using SVD

Goal: To use SVD to reconstruct a low-rank approximation of OHC

Data subset:
- North Atlantic basin
- January 1993
    - Any duplicated lat/long values have been dropped (loses 786 out of 1538 data points)

In [ ]:
import numpy as np
import pandas as pd

from plotting_functions import plot_ohc_map

## Load data

In [ ]:
df = pd.read_csv("Data/ME4OH_EN411_OFAM3/NA_1993_2014.csv")

## Preprocessing

In [ ]:
# Convert dates to datetime, save year month and year indication columns (aids plotting)
df['Date'] = pd.to_datetime(df['Date'])
df['year_month'] = df['Date'].dt.to_period('M')
df['year'] = df['Date'].dt.to_period('Y')
df

In [ ]:
# Get project type
project_type_list = []
for name in df['Project name']:
    for t in ['ARGO', 'ABSO', 'WOD', 'GTSPP']:
        if t in name:
            project_type_list.append(t)
df['Project type'] = project_type_list
df

In [ ]:
df['Project type'].value_counts()

In [ ]:
# Filter for a single month
df = df[df["year_month"]=="1993-01"]
df

In [ ]:
df['Project type'].value_counts()

In [ ]:
df['Coordinate'] = [(df.Latitude[i], df.Longitude[i]) for i in range(len(df))]
df = df.drop_duplicates('Coordinate')

In [ ]:
df['Project type'].value_counts()

In [ ]:
# Create pivot table: rows = latitude, columns = longitude, values = OHC
orig_matrix = df.pivot_table(index="Latitude", columns="Longitude", values="OHC")
# matrix = matrix.fillna(0)  # Replace NaNs with 0
matrix = orig_matrix.fillna(orig_matrix.mean(axis=1), axis=1)  # Replace with mean value for latitude
matrix

Thoughts: some of these latitudes/longitudes are on land and therefore it is unrealistic to have an OHC value here

In [ ]:
matrix.shape

In [ ]:
U, S, V = np.linalg.svd(matrix)  # automatically sorts S
U.shape, S.shape, V.shape

In [ ]:
# Adjust matrix shapes to ensure U (m x n), S (n x n), V (n x n):
U = U[:, :S.shape[0]]  # Limit U to the number of singular values
S = np.diag(S)  # Convert S (a vector) into a diagonal matrix

In [ ]:
k = 5  # Chosen rank
R = U[:, :k] @ S[:k, :k] @ V[:k, :]  # Use rank-reduced components for approximation 
R

In [ ]:
# Best k wil give this ratio~0.8 (https://www.youtube.com/watch?v=c7e-D2tmRE0)
np.linalg.norm(S[:k, :k], ord='fro')/np.linalg.norm(S, ord='fro')

In [ ]:
R.shape

In [ ]:
fro_error = np.linalg.norm(matrix - R, ord='fro')
norm_matrix = np.linalg.norm(matrix, ord='fro')
rel_error = fro_error / (norm_matrix if norm_matrix != 0 else 1.0)
rel_error

In [ ]:
fro_error

In [ ]:
reconstructed_df = pd.DataFrame(R, index=matrix.index.to_list(), columns=matrix.columns.to_list())
reconstructed_df = reconstructed_df.reset_index(drop=False, names='Latitude')
# reconstructed_df

In [ ]:
unpivot_reconstruct = reconstructed_df.melt(id_vars='Latitude', var_name='Longitude', value_name='OHC')
# unpivot_reconstruct

In [ ]:
plot_ohc_map(unpivot_reconstruct, "Reconstructed OHC", min_ohc=min(df.OHC), max_ohc=max(df.OHC), basin="NA")

### Remove values that didn't exist in original dataset

In [ ]:
# Set index and columns of original data matrix so match R
orig_matrix = orig_matrix.reset_index(drop=True)
orig_matrix.columns = range(orig_matrix.columns.size)

In [ ]:
# Mask out values that were NaN in original
R_df = pd.DataFrame(R)
R_masked = R_df.mask(orig_matrix.isnull())
# R_masked

In [ ]:
# verifications
print(orig_matrix.isnull().sum().sum() == R_masked.isnull().sum().sum()) # Check that the number of NaNs has been preserved
print(R_masked.isnull().sum().sum() != 367*455)  # Check that not all data is NaN

In [ ]:
# Reset columns and rows (index) to be lat/long values
R_masked.columns = matrix.columns.to_list()
R_masked["Latitude"] = matrix.index.to_list()
# R_masked

# Unpivot data (revert to original format, with lat, long and OHC in separate columns) + drop NaNs (masked out data points)
masked_unpivot_df = R_masked.melt(id_vars='Latitude', var_name='Longitude', value_name='OHC').dropna().reset_index(drop=True)
# masked_unpivot_df

In [ ]:
plot_ohc_map(masked_unpivot_df, "Reconstructed OHC", min_ohc=min(df.OHC), max_ohc=max(df.OHC), basin="NA")